<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 vsep — Audio Stem Separator

Welcome to the **vsep** interactive demo! Separate any audio into vocals, drums, bass, instruments, and more using 100+ state-of-the-art AI models.

## Features
- ⚡ **Fast Processing** — GPU-accelerated inference with automatic hardware detection
- 🎯 **100+ Models** — Choose from Demucs, MDX-Net, VR Band Split, and Roformer architectures
- 🎚️ **Easy to Use** — Select, separate, listen, and download in 5 steps
- 🔗 **YouTube Support** — Paste a URL and download audio directly with yt-dlp
- 📖 **[Full Wiki](https://github.com/BF667-IDLE/vsep/wiki)** for detailed docs

---

## 1️⃣ Setup

Clone the repo and install dependencies (vsep + yt-dlp). This takes ~2 minutes.

In [ ]:
#@title 1. Install vsep + yt-dlp

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
%cd /content/vsep
!pip install -q -r requirements.txt
!pip install -q yt-dlp

import os
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU detected — will use CPU (slower)")
    print("💡 Go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Get Audio

**Option A:** Upload a file from your computer.

**Option B:** Paste a YouTube / SoundCloud / Bandcamp URL and download with yt-dlp.

Run **one** of the two cells below.

In [ ]:
#@title 2A. Upload Audio File

from google.colab import files
import os

print("📁 Upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    size_mb = os.path.getsize(audio_file) / (1024 * 1024)
    print(f"✅ Uploaded: {audio_file} ({size_mb:.1f} MB)")
else:
    print("❌ No file uploaded.")

In [ ]:
#@title 2B. Download from YouTube / SoundCloud / etc.

import os, subprocess

url = "https://www.youtube.com/watch?v=EXAMPLE"  # @param {type:"string"}
format_quality = "bestaudio/best"  # @param ["bestaudio/best", "worstaudio/worst"]

if not url or "EXAMPLE" in url:
    print("❌ Please paste a valid URL above.")
else:
    print(f"🔗 Downloading audio from: {url}")
    !yt-dlp -x --audio-format wav --audio-quality 0 -o "%(title).80s.%(ext)s" --no-playlist "{url}"
    
    # Find the downloaded file
    wav_files = sorted([f for f in os.listdir(".") if f.endswith(".wav")],
                       key=lambda f: os.path.getmtime(f), reverse=True)
    if wav_files:
        audio_file = wav_files[0]
        size_mb = os.path.getsize(audio_file) / (1024 * 1024)
        print(f"✅ Downloaded: {audio_file} ({size_mb:.1f} MB)")
    else:
        print("❌ Download failed. Check the URL or try again.")

## 3️⃣ Select Model

Pick a category, then choose a model. The catalog is loaded from **models.json** + **UVR model repo** — no typing required.

💡 You can **type in the model dropdown** to search/filter within a category.

In [ ]:
#@title 3. Select Model (Categorized)

import sys, builtins
sys.path.insert(0, "/content/vsep")

from separator import Separator
from config.variables import MODEL_CATALOG, MODEL_CATEGORIES, classify_model
from ipywidgets import Dropdown, HTML, VBox, HBox
from IPython.display import display

# ── Load dynamic models from UVR ─────────────────────────────────────
sep = Separator(info_only=True)
model_list = sep.list_models()

# ── Build flat catalog: display_name → filename ───────────────────────
catalog = dict(MODEL_CATALOG)
for mtype, models in model_list.items():
    for name, info in models.items():
        catalog[name] = info["filename"]

# ── Classify every model into categories ─────────────────────────────
categorized = {}
all_cat_labels = [cat[0] for cat in MODEL_CATEGORIES] + ["📦 Other"]

for name in catalog:
    cat = classify_model(name)
    categorized.setdefault(cat, []).append(name)

# Sort models within each category alphabetically
for cat in categorized:
    categorized[cat].sort(key=lambda x: x.lower())

# Build category label with count
cat_options = ["📋 All Models"]
for label in all_cat_labels:
    if label in categorized:
        cat_options.append(f"{label}  ({len(categorized[label])})")

total = len(catalog)
print(f"  {total} models loaded")
for co in cat_options:
    print(f"    {co}")
print(f"  Type in the model dropdown to search 🔍\n")

# ── Category dropdown ─────────────────────────────────────────────────
cat_dd = Dropdown(
    options=cat_options,
    value=cat_options[0],
    description="Category:",
    layout={"width": "400px"},
)

# ── Model dropdown ────────────────────────────────────────────────────
all_sorted = sorted(catalog.keys(), key=lambda x: x.lower())
default_name = next(
    (n for n in all_sorted if "BS Roformer" in n and "EP-317" in n),
    next((n for n in all_sorted if "htdemucs_ft" in n), all_sorted[0])
)

model_dd = Dropdown(
    options=all_sorted,
    value=default_name,
    description="",",
    layout={"width": "95%"},
)

info_html = HTML("")

def update_models(*_):
    sel = cat_dd.value
    if sel == "📋 All Models":
        names = all_sorted
    else:
        # Strip the count suffix  "🎤 Vocals  (42)" → "🎤 Vocals"
        cat_label = sel.rsplit("  (")[0]
        names = sorted(categorized.get(cat_label, []), key=lambda x: x.lower())
    model_dd.options = names
    if names:
        # Try to keep the same model if it still exists
        if model_dd.value in names:
            pass
        else:
            model_dd.value = names[0]
    else:
        model_dd.options = ["(no models)"]
        model_dd.value = "(no models)"
    on_model_select({"new": model_dd.value})

def on_model_select(change):
    name = change["new"]
    if name == "(no models)":
        info_html.value = "<i>No models in this category.</i>"
        return
    fname = catalog[name]
    cat = classify_model(name)
    info_html.value = (
        f"<b>Category:</b> {cat}<br>"
        f"<b>Model:</b> {name}<br>"
        f"<b>File:</b> <code>{fname}</code>"
    )

cat_dd.observe(update_models, names="value")
model_dd.observe(on_model_select, names="value")
on_model_select({"new": default_name})

display(VBox([HBox([cat_dd]), model_dd, info_html]))

# ── Store for the next cell ───────────────────────────────────────────
builtins._vsep_catalog = catalog
builtins._vsep_dropdown = model_dd

In [ ]:
#@title 3b. Confirm Selection

selected_model = _vsep_catalog[_vsep_dropdown.value]
print(f"  ✅ {_vsep_dropdown.value}")
print(f"     → {selected_model}")

## 4️⃣ Separate Audio

Run the separation. First use may take a few minutes to download the model (cached afterwards).

In [ ]:
#@title 4. Run Separation

import sys
sys.path.insert(0, "/content/vsep")

from separator import Separator
import config.variables as cfg
import os, time

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("🚀 Starting separation...")
print(f"📁 Input: {audio_file}")
print(f"🎯 Model: {selected_model}")
print("⏳ Please wait...\n")

start = time.time()

# Initialize and run
separator = Separator(
    model_file_dir="./models",
    output_dir="./output",
    use_soundfile=True,
)

separator.load_model(selected_model)
output_files = separator.separate(audio_file)

elapsed = time.time() - start

print(f"\n✅ Separation complete! ({elapsed:.1f}s)")
if output_files:
    print(f"📂 {len(output_files)} stem(s):")
    for f in output_files:
        size = os.path.getsize(f) / (1024 * 1024)
        print(f"   — {os.path.basename(f)} ({size:.1f} MB)")
else:
    print("⚠️ No output files produced. Check the logs above for errors.")

## 5️⃣ Listen to Results

Play back the separated stems.

In [ ]:
#@title 5. Listen to Stems

import glob, os
from IPython.display import Audio, display

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"🎵 {len(output_files)} separated stem(s):\n")
    for i, fp in enumerate(output_files, 1):
        name = os.path.basename(fp)
        print(f"{i}. {name}")
        display(Audio(fp))
        print("─" * 60)
else:
    print("❌ No output files. Run the separation first.")

## 6️⃣ Download Results

Download the separated stems to your computer.

In [ ]:
#@title 6. Download Stems

from google.colab import files
import glob, os

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"📥 Downloading {len(output_files)} file(s)...\n")
    for fp in output_files:
        files.download(fp)
else:
    print("❌ No files to download. Run the separation first.")

---

## 💡 Tips & Recommendations

### Best Models by Task
| Task | Recommended Model | Why |
|------|-----------------|-----|
| **Vocals** | `BS Roformer EP-317 sdr 12.9755` | Highest SDR (12.98) of any model |
| **4-stem** | `htdemucs_ft` | Vocals + drums + bass + other |
| **6-stem** | `htdemucs_6s` | Adds piano + guitar |
| **Instrumental** | `MelBand Roformer Kim Inst V1 Plus` | Clean instrumental |
| **Karaoke** | `MelBand Roformer Karaoke (becruily)` | Aggressive vocal removal |
| **De-Reverb** | `MelBand Roformer De-Reverb (anvuew)` | Remove reverb from vocals |
| **Denoise** | `Denoise Mel Roformer Aufr33` | Clean up background noise |
| **Fast** | `MDX-Net Inst HQ 5` | Lightweight, fast inference |

### YouTube / yt-dlp Tips
- Supports **YouTube**, **SoundCloud**, **Bandcamp**, **Twitter/X**, and 1000+ more sites
- Downloads the best available audio and converts to WAV automatically
- For playlists, remove `--no-playlist` from the yt-dlp command
- To limit duration (e.g. first 10 min): add `--max-downloads 1 --match-filter "duration < 600"`

### GPU Tips
- **T4 (free)** — Good for most models. Big Roformer/Demucs may be tight on 16 GB
- **A100 (paid)** — Can run any model including the largest ones
- **Timeouts** — Sessions timeout after ~90 min of inactivity
- **Long files** — For audio > 30 min, add `chunk_duration=300` in the separation cell

### Troubleshooting

| Problem | Fix |
|---------|-----|
| **No GPU** | Runtime → Change runtime type → T4 GPU |
| **Out of memory** | Use a smaller model or shorter audio, or add `chunk_duration=300` |
| **Download failed** | Re-run the cell (downloads resume automatically) |
| **Module not found** | Re-run the Install cell or restart the runtime |
| **yt-dlp error** | The video may be age-restricted, private, or region-locked |

### Links

- 📦 **Repository:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- 📖 **Wiki:** [github.com/BF667-IDLE/vsep/wiki](https://github.com/BF667-IDLE/vsep/wiki)
- 🐛 **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)
- 📄 **License:** [MIT](https://opensource.org/licenses/MIT)

**Made with ❤️ using [vsep](https://github.com/BF667-IDLE/vsep)**